# Experiment 3: one sample, full pass and accumulation

Experiment 3 contains 96 trials: three sampling modes × four bases × four dataset partitions × two tasks. LR 3e-7, L2-SP 0.003, full parameter updates and training seed 42 are fixed.

| Mode | Data in one table visit | Optimizer updates |
|---|---|---|
| one_sample | One context/query batch | One |
| full_pass | A randomized, non-overlapping partition into batches | One per batch |
| accumulate | The same disjoint full pass | One averaged-gradient update |

All three use proportional PD sampling, unlike the balanced-PD main grid. Their one_sample control is therefore measured here rather than borrowed from experiment 1. Equal successful-update budgets do not imply equal rows or compute. Full-pass partitions are reshuffled on subsequent visits.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from IPython.display import display
from src.visualize import style
from src.visualize.figures import FigureSaver
from src.visualize import campaign as cp
from src.data.dataset_names import display_frame
from src.visualize.inputs import analysis_root
style.apply()
sink = FigureSaver('experiment3/01_sampling_comparison')
report = cp.NotebookReport('Experiment 3: one sample, full pass and accumulation')


## 1. Coverage of all three protocols

A missing sampling arm cannot be inferred from a main-sweep result. Check completed identities before comparing protocol means.

In [ ]:
runs = {track: cp.load_campaign(3,track) for track in ('pd','lgd')}
for run in runs.values():
    cp.show(sink, cp.plot_coverage(run))
    cp.show(sink, cp.plot_coverage_grid(run))
report.add('1. Protocol coverage', '\n\n'.join(cp.coverage_summary(c) for c in runs.values()))

## 2. Behavior at equal update budgets

At most three curves appear per base. Positive effects are improvements relative to the same dataset at update zero. Curves with missing baseline observations at a milestone retain gaps.

In [ ]:
for run in runs.values():
    cp.show(sink, cp.plot_sampling_trajectories(run))
report.add('2. Equal-update behavior', '\n\n'.join(c.track.upper()+'\n'+cp.effect_summary(cp.effects(c)) for c in runs.values()))

## 3. The same behavior against row exposure

An accumulation update visits many more rows than a one-sample update. These axes reveal exposure differences without pretending to identify unique examples or equal-compute effects.

In [ ]:
for run in runs.values():
    cp.show(sink, cp.plot_sampling_trajectories(run,row_exposure=True))
report.add('3. Row exposure', 'Horizontal positions are median processed row exposures over contributing trials, not unique rows. Update-budget and exposure comparisons answer different questions.')

## 4. Matched endpoint differences

Compare each full-pass protocol to the proportional one-sample control on the same dataset, base and settings. Positive contrast favors the alternative.

In [ ]:
for run in runs.values():
    endpoint = cp.endpoint_effects(run)
    for mode in ('full_pass','accumulate'):
        contrast = cp.factor_contrasts(endpoint,'sampling','one_sample',mode)
        cp.show(sink, cp.plot_contrasts(contrast,run.metric,mode+' minus one_sample'))
        report.add('4. '+run.track.upper()+' / '+mode, cp.effect_summary(contrast,'contrast'))

## 5. What did each protocol cost?

Compare seconds per update, processed exposures and allocated GPU hours. These quantities explain why an update-matched comparison alone cannot establish computational efficiency.

In [ ]:
for run in runs.values():
    cp.show(sink, cp.plot_sampling_cost(run))
report.add('5. Computing cost', 'Costs use completed trials only; pending/failed identities remain in the coverage section. Accumulation can be more expensive per update while giving tables equal update counts.')

## 6. Final benchmark protocol effects

Repeat the matched contrasts on complete held-out benchmark folds. This comparison is kept separate from monitoring.

In [ ]:
for run in runs.values():
    benchmark = cp.benchmark_effects(run)
    cp.show(sink, cp.plot_eval_coverage(run))
    for mode in ('full_pass','accumulate'):
        contrast = cp.factor_contrasts(benchmark,'sampling','one_sample',mode)
        cp.show(sink, cp.plot_contrasts(contrast,run.metric,'benchmark '+mode+' minus one_sample'))
        report.add('6. '+run.track.upper()+' / '+mode, cp.effect_summary(contrast,'contrast'))

## 7. Interpretation

This experiment changes gradient aggregation and row coverage while retaining a fixed successful-update budget. It does not isolate those mechanisms at equal GPU time. Balanced versus proportional PD context sampling is also distinct from this three-mode comparison.

In [ ]:
report.add('7. Interpretation limits', 'Within-experiment protocol comparison only. Do not treat experiment-1 balanced-PD one_sample as the experiment-3 proportional control, or equal updates as equal compute.')

## Summary

The following text repeats the sections in order. It is included verbatim in `All_Results.md`.

In [ ]:
print(report.summary(sink))